# 🧠 Brain Tumor Classification from MRI Images
### *Image Processing & Computer Vision Interactive Project Walkthrough*

This notebook demonstrates every phase of the project pipeline:
1. **Exploratory Data Analysis & MRI Visualization**
2. **Classical Image Processing:** Skull Stripping via Otsu Thresholding & Extreme Contour Extraction
3. **Contrast Enhancement:** CLAHE (Contrast Limited Adaptive Histogram Equalization)
4. **Handcrafted Feature Extraction:** GLCM (Haralick Texture Descriptors) & Local Binary Patterns (LBP)
5. **Deep Learning Model Inference & Explainability:** Grad-CAM Saliency Heatmaps

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

# Set project root
sys.path.append(str(Path.cwd().parent))
from src.config import CLASSES, IMG_SIZE, DEVICE
from src.image_processing.skull_stripping import crop_brain_contour
from src.image_processing.enhancement import apply_clahe
from src.image_processing.feature_extraction import extract_glcm_features, extract_lbp_features
from src.models.custom_cnn import BrainTumorCNN
from src.explainability.gradcam import GradCAM, overlay_cam_on_image

print(f"Environment verified. Execution Device: {DEVICE}")

## 1. Classical Image Processing: Skull Stripping & Enhancement
We load a sample MRI scan, apply Gaussian filtering, Otsu's thresholding, and morphological operations to extract extreme contour points and crop the cerebral region.

In [ ]:
# Load sample test image generated by sanity run
sample_path = Path.cwd().parent / "outputs" / "test_run" / "sample_brain_mri.png"
raw_img = cv2.imread(str(sample_path))

# Apply contour crop and CLAHE
cropped_img = crop_brain_contour(raw_img)
enhanced_img = apply_clahe(cropped_img, clip_limit=2.0)

# Visualize progression
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("1. Original MRI Scan")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("2. Skull-Stripped (Extreme Contours)")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(enhanced_img, cv2.COLOR_BGR2RGB))
axes[2].set_title("3. Contrast Enhanced (CLAHE)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 2. Handcrafted Texture Extraction (GLCM Haralick & LBP)
Classical Computer Vision features extracted for baseline machine learning comparisons.

In [ ]:
glcm_metrics = extract_glcm_features(enhanced_img)
print("--- Haralick GLCM Texture Features ---")
for k, v in glcm_metrics.items():
    print(f"{k.capitalize():15s}: {v:.4f}")

## 3. Explainable AI: Grad-CAM Saliency Heatmap
Visualizes the convolutional activation map to verify lesion localization.

In [ ]:
# Load CNN model
model = BrainTumorCNN(num_classes=len(CLASSES)).to(DEVICE)
model.eval()

# Forward pass
resized = cv2.resize(enhanced_img, IMG_SIZE)
tensor = torch.tensor(resized, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE) / 255.0

# Generate Grad-CAM
target_layer = model.features[-1].block[0]
cam_gen = GradCAM(model, target_layer)
cam = cam_gen.generate_heatmap(tensor, target_class=0)
overlay = overlay_cam_on_image(resized, cam)

plt.figure(figsize=(6, 6))
plt.imshow(overlay)
plt.title("Grad-CAM Class Activation Map (Saliency Overlay)")
plt.axis("off")
plt.show()